In [1]:
import pandas as pd
import numpy as np
import torch

from _ctgan.synthesizer import _CTGANSynthesizer
from table_evaluator import TableEvaluator



C:\Users\Aditya\Desktop\VECC Intern\AI NIDS Project\New folder\_ctgan\synthesizer.py:7: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


IPython not installed.


In [2]:
df = pd.read_csv("test-cse-cic-ids2018-preprocessed.csv")

print("Original shape:", df.shape)

df = df.sample(frac=0.05, random_state=42)
print("5% shape:", df.shape)

C:\Users\Aditya\AppData\Local\Temp\ipykernel_3436\4076556315.py:1: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("test-cse-cic-ids2018-preprocessed.csv")


Original shape: (16233002, 68)
5% shape: (811650, 68)


In [ ]:
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()
df = df.drop_duplicates()

label_col = "Label"


In [ ]:

df[label_col] = df[label_col].astype(str).str.strip()
df = df[df[label_col].str.lower() != "benign"]

print("After removing Benign:", df.shape)
print(df[label_col].value_counts())


In [ ]:
if "Protocol" not in df.columns:
    raise ValueError("Protocol column not found")

df["Protocol"] = df["Protocol"].astype(str).str.strip()



In [ ]:

print("Final real data shape:", df.shape)
print(df[label_col].value_counts())
print(df["Protocol"].value_counts())



In [ ]:

discrete_columns = ["Protocol", label_col]


In [ ]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ctgan = _CTGANSynthesizer(
    embedding_dim=128,
    gen_dim=(256, 256),
    dis_dim=(256, 256),
    l2scale=1e-6,
    batch_size=500,
    patience=25
)

ctgan.fit(
    train_data=df_small,
    discrete_columns=discrete_columns,
    epochs=100
)


In [ ]:

synthetic_df = ctgan.sample(50000)

print("Synthetic shape:", synthetic_df.shape)
print(synthetic_df.head())
print(synthetic_df[label_col].value_counts())
print(synthetic_df["Protocol"].value_counts())

In [ ]:
synthetic_df.to_csv(
    "cicids2018_ctgan_synthetic_nonbenign_protocol_01.csv",
    index=False
)

print("Saved: cicids2018_ctgan_synthetic_nonbenign_protocol_01.csv")


In [ ]:
print("\nRunning TableEvaluator...")

# Use same number of rows for fair comparison
n_eval = min(len(df_small), len(synthetic_df), 50000)

real_eval = df_small.sample(n=n_eval, random_state=42).reset_index(drop=True)
fake_eval = synthetic_df.sample(n=n_eval, random_state=42).reset_index(drop=True)

# Make sure column order is same
fake_eval = fake_eval[real_eval.columns]

# Convert categorical columns to string
for col in discrete_columns:
    real_eval[col] = real_eval[col].astype(str)
    fake_eval[col] = fake_eval[col].astype(str)

table_evaluator = TableEvaluator(
    real_eval,
    fake_eval,
    cat_cols=discrete_columns
)

# table_evaluator.visual_evaluation()

results = table_evaluator.evaluate(
    target_col=label_col
)

print("\nTableEvaluator Results:")
print(results)